# UNGRADED Workbook for In-Class

This notebook is here for you to "code along" during class. 

It will not be graded, so feel free to play around!

In [7]:
# Load and preview data
import pandas as pd
import requests

url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json"
data = pd.read_json(url)
print(data.head())


               time_tag  satellite          flux  observed_flux  \
0  2025-05-05T23:07:00Z         19  1.000000e-09   5.841829e-08   
1  2025-05-05T23:07:00Z         19  9.385262e-07   1.021699e-06   
2  2025-05-05T23:08:00Z         19  1.000000e-09   5.764739e-08   
3  2025-05-05T23:08:00Z         19  9.405587e-07   1.022420e-06   
4  2025-05-05T23:09:00Z         19  1.000000e-09   6.049294e-08   

   electron_correction  electron_contaminaton      energy  
0         6.057938e-08                   True  0.05-0.4nm  
1         8.317302e-08                  False   0.1-0.8nm  
2         5.850535e-08                   True  0.05-0.4nm  
3         8.186085e-08                  False   0.1-0.8nm  
4         5.955845e-08                   True  0.05-0.4nm  


In [ ]:
data['time_tag'] = pd.to_datetime(data['time_tag'])
print(data.describe())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(data['time_tag'], data['flux'], label='Short', alpha=0.7)
plt.plot(data['time_tag'], data['flux_l'], label='Long', alpha=0.7)
plt.yscale('log')
plt.xlabel('Time')
plt.ylabel('Flux (log scale)')
plt.legend()
plt.title('GOES X-ray Flux Over 24 Hours')
plt.show()


In [ ]:
print(data.columns)


In [ ]:
# Inspect sample rows
print(data.head())

# Pivot to get separate flux values for short and long wavelengths
pivoted = data.pivot(index='time_tag', columns='energy', values='flux')

# Check the new DataFrame
print(pivoted.columns)


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(pivoted.index, pivoted['0.05-0.4nm'], label='Short (0.05-0.4nm)', alpha=0.7)
plt.plot(pivoted.index, pivoted['0.1-0.8nm'], label='Long (0.1-0.8nm)', alpha=0.7)
plt.yscale('log')
plt.xlabel('Time')
plt.ylabel('Flux (Watts/m², log scale)')
plt.legend()
plt.title('GOES X-ray Flux Over 24 Hours')
plt.tight_layout()
plt.show()


In [ ]:
flare_threshold = 1e-5  # Example threshold for M-class flares
flares = pivoted[pivoted['0.1-0.8nm'] > flare_threshold]
print(flares)


In [ ]:
# Extract hour
pivoted['hour'] = pivoted.index.hour
hourly_max = pivoted.groupby('hour').max()

# Heatmap
import seaborn as sns
sns.heatmap(hourly_max[['0.1-0.8nm']], cmap='inferno', annot=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data
url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json"
data = pd.read_json(url)

# Convert time_tag to datetime
data['time_tag'] = pd.to_datetime(data['time_tag'])

# Pivot data to get short and long wavelengths
pivoted = data.pivot(index='time_tag', columns='energy', values='flux')

# Rename columns for easier reference
pivoted.columns = ['Short (0.05-0.4nm)', 'Long (0.1-0.8nm)']
pivoted = pivoted.dropna()

# 1. Time Series Plot with log scale
plt.figure(figsize=(14, 6))
plt.plot(pivoted.index, pivoted['Short (0.05-0.4nm)'], label='Short (0.05-0.4nm)', alpha=0.7)
plt.plot(pivoted.index, pivoted['Long (0.1-0.8nm)'], label='Long (0.1-0.8nm)', alpha=0.7)
plt.yscale('log')
plt.axhline(1e-6, color='orange', linestyle='--', linewidth=0.8, label='C-class Threshold')
plt.axhline(1e-5, color='red', linestyle='--', linewidth=0.8, label='M-class Threshold')
plt.axhline(1e-4, color='purple', linestyle='--', linewidth=0.8, label='X-class Threshold')
plt.xlabel('Time')
plt.ylabel('Flux (Watts/m², log scale)')
plt.title('GOES X-ray Flux Over 24 Hours')
plt.legend()
plt.tight_layout()
plt.show()

# 2. Flare Classification
def classify_flare(flux):
    if flux < 1e-7:
        return 'A'
    elif flux < 1e-6:
        return 'B'
    elif flux < 1e-5:
        return 'C'
    elif flux < 1e-4:
        return 'M'
    else:
        return 'X'

flare_classes = pivoted['Long (0.1-0.8nm)'].apply(classify_flare)
flare_counts = flare_classes.value_counts().sort_index()

# Bar Chart of Flare Classes
plt.figure(figsize=(8, 5))
sns.barplot(x=flare_counts.index, y=flare_counts.values, palette='magma')
plt.title('Flare Class Distribution (Based on 0.1–0.8 nm Flux)')
plt.xlabel('Flare Class')
plt.ylabel('Number of Observations')
plt.tight_layout()
plt.show()

# 3. Heatmap of Maximum Hourly Flux
pivoted['hour'] = pivoted.index.hour
hourly_max = pivoted.groupby('hour').max()

plt.figure(figsize=(8, 6))
sns.heatmap(hourly_max[['Long (0.1-0.8nm)']], cmap='inferno', annot=True, fmt=".1e")
plt.title('Hourly Max Flux (Long Wavelength)')
plt.xlabel('Hour of Day (UTC)')
plt.ylabel('')
plt.tight_layout()
plt.show()

# 4. Correlation between Short and Long
plt.figure(figsize=(7, 6))
sns.scatterplot(x=pivoted['Short (0.05-0.4nm)'], y=pivoted['Long (0.1-0.8nm)'], alpha=0.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Short (0.05–0.4 nm) Flux')
plt.ylabel('Long (0.1–0.8 nm) Flux')
plt.title('Correlation between Short and Long Wavelengths')
plt.tight_layout()
plt.show()


In [ ]:
!pip install cartopy


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests
from datetime import datetime

# Load JSON data from URL
url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json"
response = requests.get(url)
data = pd.DataFrame(response.json())

# Convert time_tag to datetime
data['time_tag'] = pd.to_datetime(data['time_tag'])

# Filter only long wavelength data
long_flux = data[data['energy'] == '0.1-0.8nm'].copy()

# Get strongest flare
strongest = long_flux.loc[long_flux['flux'].idxmax()]
flare_time = strongest['time_tag']
flare_flux = strongest['flux']

# Convert flare time to subsolar longitude
utc_hour = flare_time.hour + flare_time.minute / 60
subsolar_lon = -15 * utc_hour  # Earth rotates 15° per hour

# Plot map
plt.figure(figsize=(14, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.coastlines()
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')

# Draw daylight region as shaded box
impact_lon_min = subsolar_lon - 60
impact_lon_max = subsolar_lon + 60
ax.add_patch(plt.Rectangle((impact_lon_min, -60), 120, 120,
                           transform=ccrs.PlateCarree(), color='orange', alpha=0.3,
                           label='Sunlit Impact Zone'))

# Add annotation
plt.title(f'Solar X-ray Flare Impact Projection\nStrongest Flare: {flare_time} UTC (Flux: {flare_flux:.1e})')
plt.legend(loc='lower left')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests
import numpy as np
from datetime import datetime

# Load data
url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json"
response = requests.get(url)
data = pd.DataFrame(response.json())
data['time_tag'] = pd.to_datetime(data['time_tag'])

# Long band only
long_flux = data[data['energy'] == '0.1-0.8nm'].copy()

# Flare color by class
def flare_color(flux):
    if flux < 1e-6:
        return 'yellow'
    elif flux < 1e-5:
        return 'orange'
    elif flux < 1e-4:
        return 'red'
    else:
        return 'purple'

# Plot map + time series
fig = plt.figure(figsize=(16, 7))
gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1])

# --- MAP ---
ax_map = plt.subplot(gs[0], projection=ccrs.PlateCarree())
ax_map.set_global()
ax_map.coastlines()
ax_map.add_feature(cfeature.BORDERS)
ax_map.add_feature(cfeature.LAND, facecolor='lightgray')
ax_map.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax_map.set_title('Global Map of Flare Impacts')

# Plot flares
strong_flares = long_flux[long_flux['flux'] > 1e-6]
for _, row in strong_flares.iterrows():
    t = row['time_tag']
    flux = row['flux']
    lon = -15 * (t.hour + t.minute / 60)
    color = flare_color(flux)
    size = min(flux * 1e8, 100)
    ax_map.plot(lon, 0, marker='o', color=color, markersize=size,
                alpha=0.5, transform=ccrs.PlateCarree())

# Highlight strongest flare
strongest = strong_flares.loc[strong_flares['flux'].idxmax()]
t_max = strongest['time_tag']
subsolar_lon = -15 * (t_max.hour + t_max.minute / 60)
ax_map.add_patch(plt.Rectangle((subsolar_lon - 60, -60), 120, 120,
                transform=ccrs.PlateCarree(), color='orange', alpha=0.2,
                label='Sunlit Impact Zone'))
ax_map.legend(loc='lower left')

# --- TIME SERIES ---
ax_time = plt.subplot(gs[1])
ax_time.plot(long_flux['time_tag'], long_flux['flux'], color='black')
ax_time.set_yscale('log')
ax_time.set_title('Flare Flux Time Series')
ax_time.set_xlabel('UTC Time')
ax_time.set_ylabel('Flux (W/m²)')
ax_time.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax_time.axhline(1e-6, color='orange', linestyle='--', label='C-class')
ax_time.axhline(1e-5, color='red', linestyle='--', label='M-class')
ax_time.axhline(1e-4, color='purple', linestyle='--', label='X-class')
ax_time.legend()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests
import numpy as np
from datetime import datetime

# Load data from NOAA
url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json"
response = requests.get(url)
data = pd.DataFrame(response.json())

# Preprocessing
data['time_tag'] = pd.to_datetime(data['time_tag'])
long_flux = data[data['energy'] == '0.1-0.8nm'].copy()

# Sort by flux and take top 10 flares
strong_flares = long_flux.sort_values(by='flux', ascending=False).head(10)

# Define flare class colors
def flare_color(flux):
    if flux < 1e-6:
        return 'yellow'
    elif flux < 1e-5:
        return 'orange'
    elif flux < 1e-4:
        return 'red'
    else:
        return 'purple'

# Create side-by-side layout
fig = plt.figure(figsize=(16, 7))
gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1])

# --------- MAP PLOT ----------
ax_map = plt.subplot(gs[0], projection=ccrs.PlateCarree())
ax_map.set_global()
ax_map.coastlines()
ax_map.gridlines(draw_labels=True)
ax_map.add_feature(cfeature.LAND, facecolor='lightgray')
ax_map.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax_map.set_title('🌍 Flare Impact Projection (Top 10 Events)', fontsize=14)

# Plot top 10 flare markers
for _, row in strong_flares.iterrows():
    flare_time = row['time_tag']
    flux = row['flux']
    lon = -15 * (flare_time.hour + flare_time.minute / 60)
    color = flare_color(flux)
    size = np.clip(flux * 1e8, 10, 40)
    ax_map.plot(lon, 0, marker='o', color=color, markersize=size, 
                transform=ccrs.PlateCarree(), alpha=0.6)

# Draw sunlit region for strongest flare
strongest = strong_flares.iloc[0]
strongest_time = strongest['time_tag']
subsolar_lon = -15 * (strongest_time.hour + strongest_time.minute / 60)
ax_map.add_patch(plt.Rectangle((subsolar_lon - 60, -60), 120, 120,
    transform=ccrs.PlateCarree(), color='gold', alpha=0.2, label='Approx. Daylit Region'))
ax_map.legend(loc='lower left')

# --------- TIME SERIES PLOT ----------
ax_time = plt.subplot(gs[1])
ax_time.plot(long_flux['time_tag'], long_flux['flux'], color='black', label='X-ray Flux (0.1–0.8nm)', linewidth=1)
ax_time.set_yscale('log')
ax_time.set_title('📈 Flare Flux Over Time (log scale)', fontsize=14)
ax_time.set_xlabel('UTC Time')
ax_time.set_ylabel('Flux (W/m²)')
ax_time.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Highlight class lines
ax_time.axhline(1e-6, color='orange', linestyle='--', linewidth=1, label='C-class')
ax_time.axhline(1e-5, color='red', linestyle='--', linewidth=1, label='M-class')
ax_time.axhline(1e-4, color='purple', linestyle='--', linewidth=1, label='X-class')
ax_time.legend(loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
!pip install ipywidgets


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests
import numpy as np

# -----------------------------------------------
# 🛠️ CONFIG: Toggle Aurora Overlay
show_aurora = True  # Change to False to hide aurora zones
# -----------------------------------------------

# 📦 Load data from NOAA GOES
url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json"
response = requests.get(url)
data = pd.DataFrame(response.json())

# 🧹 Preprocessing
data['time_tag'] = pd.to_datetime(data['time_tag'])
long_flux = data[data['energy'] == '0.1-0.8nm'].copy()
strong_flares = long_flux.sort_values(by='flux', ascending=False).head(10)

# 🎨 Flare class colors
def flare_color(flux):
    if flux < 1e-6:
        return 'yellow'
    elif flux < 1e-5:
        return 'orange'
    elif flux < 1e-4:
        return 'red'
    else:
        return 'purple'

# 📊 Set up figure
fig = plt.figure(figsize=(16, 7))
gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1])

# -------- MAP PLOT --------
ax_map = plt.subplot(gs[0], projection=ccrs.PlateCarree())
ax_map.set_global()
ax_map.coastlines()
ax_map.gridlines(draw_labels=True)
ax_map.add_feature(cfeature.LAND, facecolor='lightgray')
ax_map.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax_map.set_title('🌍 Flare Impact Map (Top 10 Events)', fontsize=14)

# 🔴 Plot top flare markers
for _, row in strong_flares.iterrows():
    flare_time = row['time_tag']
    flux = row['flux']
    lon = -15 * (flare_time.hour + flare_time.minute / 60)
    color = flare_color(flux)
    size = np.clip(flux * 1e8, 10, 40)
    ax_map.plot(lon, 0, marker='o', color=color, markersize=size,
                transform=ccrs.PlateCarree(), alpha=0.6)

# ☀️ Add daylit region
strongest = strong_flares.iloc[0]
strongest_time = strongest['time_tag']
subsolar_lon = -15 * (strongest_time.hour + strongest_time.minute / 60)
ax_map.add_patch(plt.Rectangle((subsolar_lon - 60, -60), 120, 120,
                               transform=ccrs.PlateCarree(), color='gold', alpha=0.2,
                               label='Daylit Region'))

# 🌀 Optional: Aurora Zones
if show_aurora:
    for lat in [65, -65]:
        ax_map.plot(np.linspace(-180, 180, 500), [lat]*500,
                    transform=ccrs.Geodetic(), linestyle='--',
                    color='blue', linewidth=1.2,
                    label='Aurora Oval' if lat == 65 else None)

ax_map.legend(loc='lower left')

# -------- TIME SERIES PLOT --------
ax_time = plt.subplot(gs[1])
ax_time.plot(long_flux['time_tag'], long_flux['flux'], color='black',
             label='X-ray Flux (0.1–0.8nm)', linewidth=1)
ax_time.set_yscale('log')
ax_time.set_title('📈 X-ray Flux Time Series (log scale)', fontsize=14)
ax_time.set_xlabel('UTC Time')
ax_time.set_ylabel('Flux (W/m²)')
ax_time.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Flare class thresholds
ax_time.axhline(1e-6, color='orange', linestyle='--', linewidth=1, label='C-class')
ax_time.axhline(1e-5, color='red', linestyle='--', linewidth=1, label='M-class')
ax_time.axhline(1e-4, color='purple', linestyle='--', linewidth=1, label='X-class')
ax_time.legend(loc='upper left')

plt.tight_layout()
plt.show()


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests
import numpy as np
from ipywidgets import interact, IntSlider, Checkbox

# Load the data from NOAA GOES API
url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json"
response = requests.get(url)
data = pd.DataFrame(response.json())

# Preprocessing
data['time_tag'] = pd.to_datetime(data['time_tag'])
long_flux = data[data['energy'] == '0.1-0.8nm'].copy()
long_flux_sorted = long_flux.sort_values(by='flux', ascending=False)

# Define flare class color
def flare_color(flux):
    if flux < 1e-6:
        return 'yellow'
    elif flux < 1e-5:
        return 'orange'
    elif flux < 1e-4:
        return 'red'
    else:
        return 'purple'

# Main interactive plotting function
def plot_interactive_flare_map(top_n=5, show_aurora=True):
    top_flares = long_flux_sorted.head(top_n)

    fig = plt.figure(figsize=(16, 7))
    gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1])

    # --- MAP PLOT ---
    ax_map = plt.subplot(gs[0], projection=ccrs.PlateCarree())
    ax_map.set_global()
    ax_map.coastlines()
    ax_map.gridlines(draw_labels=True)
    ax_map.add_feature(cfeature.LAND, facecolor='lightgray')
    ax_map.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax_map.set_title(f' Top {top_n} Flare Impact Map', fontsize=14)

    for _, row in top_flares.iterrows():
        flare_time = row['time_tag']
        flux = row['flux']
        lon = -15 * (flare_time.hour + flare_time.minute / 60)
        color = flare_color(flux)
        size = np.clip(flux * 1e8, 10, 40)
        ax_map.plot(lon, 0, marker='o', color=color, markersize=size,
                    transform=ccrs.PlateCarree(), alpha=0.6)

    # --- SUNLIT REGION ---
    strongest = top_flares.iloc[0]
    subsolar_lon = -15 * (strongest['time_tag'].hour + strongest['time_tag'].minute / 60)
    ax_map.add_patch(plt.Rectangle((subsolar_lon - 60, -60), 120, 120,
                                   transform=ccrs.PlateCarree(), color='gold', alpha=0.2,
                                   label='Daylit Region'))

    # --- AURORA RINGS ---
    if show_aurora:
        for lat in [65, -65]:
            ax_map.plot(np.linspace(-180, 180, 500), [lat]*500,
                        transform=ccrs.Geodetic(), linestyle='--',
                        color='blue', linewidth=1.2,
                        label='Aurora Oval' if lat == 65 else None)

    ax_map.legend(loc='lower left')

    # --- TIME SERIES PLOT ---
    ax_time = plt.subplot(gs[1])
    ax_time.plot(long_flux['time_tag'], long_flux['flux'], color='black',
                 label='X-ray Flux (0.1–0.8nm)', linewidth=1)
    ax_time.set_yscale('log')
    ax_time.set_title(' X-ray Flux Time Series (log scale)', fontsize=14)
    ax_time.set_xlabel('UTC Time')
    ax_time.set_ylabel('Flux (W/m²)')
    ax_time.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

    # Threshold lines
    ax_time.axhline(1e-6, color='orange', linestyle='--', linewidth=1, label='C-class')
    ax_time.axhline(1e-5, color='red', linestyle='--', linewidth=1, label='M-class')
    ax_time.axhline(1e-4, color='purple', linestyle='--', linewidth=1, label='X-class')
    ax_time.legend(loc='upper left')

    plt.tight_layout()
    plt.show()

# Activate interactivity
interact(plot_interactive_flare_map,
         top_n=IntSlider(value=5, min=1, max=20, step=1, description='Top N Flares'),
         show_aurora=Checkbox(value=True, description='Show Aurora Zones'));


interactive(children=(IntSlider(value=5, description='Top N Flares', max=20, min=1), Checkbox(value=True, desc…